# What text looks like after tokenization

A tokenizer converts text into integer IDs. The model never receives Python strings directly.

```text
text -> token pieces -> token IDs -> embedding vectors -> Transformer
```

This notebook uses a tiny byte-level subword tokenizer so every step is visible. Its vocabulary and IDs are illustrative; a real model's tokenizer will produce different pieces and IDs.

## 1. A minimal byte-level subword tokenizer

Production BPE tokenizers begin with bytes and learn frequent byte sequences to store as larger tokens. Here we provide a few already-learned pieces and greedily select the longest matching piece. Any unmatched text falls back to individual UTF-8 bytes, so every input remains representable.

In [ ]:
import platform

import torch
from torch import nn


class TinyByteTokenizer:
    def __init__(self):
        learned_pieces = [b"Hello", b",", b" token", b"ization", b"!"]
        self.piece_to_id = {piece: 256 + i for i, piece in enumerate(learned_pieces)}
        self.id_to_piece = {token_id: piece for piece, token_id in self.piece_to_id.items()}
        self.pieces_by_length = sorted(learned_pieces, key=len, reverse=True)
        self.pad_id = 261
        self.vocab_size = 262

    def encode(self, text):
        data = text.encode("utf-8")
        ids = []
        position = 0
        while position < len(data):
            piece = next(
                (p for p in self.pieces_by_length if data.startswith(p, position)),
                None,
            )
            if piece is None:
                ids.append(data[position])
                position += 1
            else:
                ids.append(self.piece_to_id[piece])
                position += len(piece)
        return ids

    def pieces(self, ids):
        result = []
        for token_id in ids:
            piece = self.id_to_piece[token_id] if token_id in self.id_to_piece else bytes([token_id])
            result.append(piece.decode("utf-8", errors="replace"))
        return result

    def decode(self, ids):
        data = b"".join(
            self.id_to_piece[i] if i in self.id_to_piece else bytes([i])
            for i in ids
        )
        return data.decode("utf-8")


tokenizer = TinyByteTokenizer()
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | CPU")

## 2. Input and output

The leading space belongs to the `" token"` piece. Spaces are commonly stored inside tokens because `"token"` and `" token"` occur in different contexts. `tokenization` is split into two reusable subwords.

In [ ]:
text = "Hello, tokenization!"
input_ids = tokenizer.encode(text)

print("Input text:    ", repr(text))
print("Token pieces: ", [repr(piece) for piece in tokenizer.pieces(input_ids)])
print("input_ids:    ", input_ids)
print("attention_mask", [1] * len(input_ids))
print("Decoded text:  ", repr(tokenizer.decode(input_ids)))

assert tokenizer.decode(input_ids) == text

## 3. A padded batch

Sequences in one tensor need the same length. The tokenizer pads shorter sequences and produces an attention mask: `1` means real input and `0` means padding.

In [ ]:
texts = ["Hello, tokenization!", "Hello!"]
encoded = [tokenizer.encode(item) for item in texts]
max_length = max(map(len, encoded))

padded_ids = [ids + [tokenizer.pad_id] * (max_length - len(ids)) for ids in encoded]
attention_mask = [[1] * len(ids) + [0] * (max_length - len(ids)) for ids in encoded]

input_ids = torch.tensor(padded_ids)
attention_mask = torch.tensor(attention_mask)

print("input_ids =\n", input_ids)
print("attention_mask =\n", attention_mask)
print("shape:", tuple(input_ids.shape), "= [batch, sequence]")

## 4. What enters the Transformer

The tokenizer stops at integer IDs and masks. The model's embedding table converts each ID into a continuous vector.

In [ ]:
embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim=8)
hidden_states = embedding(input_ids)

print("Token ID shape:  ", tuple(input_ids.shape))
print("Embedding shape: ", tuple(hidden_states.shape), "= [batch, sequence, hidden]")

## Takeaway

After tokenization, text is a sequence or padded batch of integer IDs plus masks. Chat templates and special role tokens are normally added before or during this step. During response-only SFT, prompt IDs remain in the input context but their label positions are commonly set to `-100`, so loss is calculated only on assistant tokens.

Continue with [Tokenizer design and determinism](tokenizer-design-and-determinism.md) and [Exact token trajectories in RL and OPD](exact-token-trajectories-in-rl-and-opd.md).